In [1]:
import sys
import ast
import glob
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.tri import Triangulation

from threshold_fit import locate_threshold, fit_threshold

%load_ext autoreload
%autoreload 2

In [ ]:
CSV_GLOB = '../out/collection/threshold_3d,*.csv'

files = sorted(glob.glob(CSV_GLOB))
if not files:
    raise FileNotFoundError(f'No files matched: {CSV_GLOB}')
print(f'Loading {len(files)} file(s):')
for f in files:
    print(f'  {f}')

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Merge repeated runs by strong_id (same circuit + decoder).
df = df.groupby('strong_id').agg({
    'shots': 'sum',
    'errors': 'sum',
    'discards': 'sum',
    'seconds': 'sum',
    'decoder': 'first',
    'json_metadata': 'first',
}).reset_index()

# Translate legacy 'p_depolarizing' / 'ratio_depolarizing' / 'depolarizing-limited'
# to the new names so old CSVs stay readable after the rename.
def _translate(meta):
    m = ast.literal_eval(meta) if isinstance(meta, str) else dict(meta)
    if 'p_depolarizing' in m and 'p_res' not in m:
        m['p_res'] = m.pop('p_depolarizing')
    if 'ratio_depolarizing' in m and 'ratio_res' not in m:
        m['ratio_res'] = m.pop('ratio_depolarizing')
    if m.get('direction') == 'depolarizing-limited':
        m['direction'] = 'res-limited'
    return m

meta = df['json_metadata'].apply(_translate)
df['json_metadata'] = meta  # keep the translated dict for downstream sinter.TaskStats
meta_df = pd.json_normalize(meta)
df = pd.concat([df, meta_df], axis=1)

print(f'\n{len(df)} merged rows. Directions: {sorted(df["direction"].unique())}')
df[['direction', 'p', 'expected_distance', 'rounds', 'shots', 'errors']].head()

In [ ]:
# Fit a threshold per direction (one ratio triple).
#
# Same pipeline as threshold_comparison.ipynb cell 24a2b34b: convert per-circuit
# logical error to per-d-rounds single-observable rate, then run the
# locate -> quadratic-FSS fit from src/threshold_fit.py.

N_PER_SIDE = 2          # 5-point window around p_init
LOGICAL_CUTOFF = 0.2    # drop saturated points

direction_keys = ['ratio_m_bulk', 'ratio_spam', 'ratio_res']

fit_rows = []
for (rm, rs, rd), grp in df.groupby(direction_keys):
    direction_name = grp['direction'].iloc[0]
    p_arr = grp['p'].to_numpy(dtype=float)
    d_arr = grp['expected_distance'].to_numpy(dtype=float)
    rounds_arr = grp['rounds'].to_numpy(dtype=float)
    k_arr = grp['k'].to_numpy(dtype=float)
    shots_arr = grp['shots'].to_numpy(dtype=float)
    errors_arr = grp['errors'].to_numpy(dtype=float)

    valid = (shots_arr > 0) & (errors_arr > 0) & (errors_arr < shots_arr * LOGICAL_CUTOFF)
    pL_total = np.where(valid, errors_arr / np.maximum(shots_arr, 1), np.nan)
    sigma_total = np.where(
        valid,
        np.sqrt(np.clip(pL_total * (1 - pL_total), 0, None) / np.maximum(shots_arr, 1)),
        np.nan,
    )

    scale = d_arr / (rounds_arr * k_arr)
    pL_per_d = 1 - (1 - pL_total) ** scale
    sigma_per_d = sigma_total * scale * np.power(np.clip(1 - pL_total, 1e-300, 1), scale - 1)

    p_init = locate_threshold(p_arr, d_arr, pL_per_d)
    if p_init is None:
        print(f'{direction_name:>22s}: could not locate threshold')
        continue
    try:
        fit = fit_threshold(
            p_arr, d_arr, pL_per_d,
            sigma=sigma_per_d, p_init=p_init, n_per_side=N_PER_SIDE,
        )
    except Exception as e:
        print(f'{direction_name:>22s}: fit failed - {e}')
        continue

    fit_rows.append({
        'direction': direction_name,
        'ratio_m_bulk': rm,
        'ratio_spam': rs,
        'ratio_res': rd,
        'p_th': fit.p_th,
        'p_th_err': fit.p_th_err,
        'mu': fit.mu,
        'mu_err': fit.mu_err,
        'p_m_bulk_th': fit.p_th * rm,
        'p_spam_th': fit.p_th * rs,
        'p_res_th': fit.p_th * rd,
        'n_points': fit.n_points,
    })
    print(
        f'{direction_name:>22s}: p_th = {fit.p_th:.4f} +/- {fit.p_th_err:.4f}  '
        f'mu = {fit.mu:.2f} +/- {fit.mu_err:.2f}  (n={fit.n_points})'
    )

fits_df = pd.DataFrame(fit_rows)
fits_df

In [ ]:
# Logical-error-rate curves per direction, with the fitted threshold drawn
# as a vertical dashed line. Same plotting helper as threshold_comparison.ipynb
# but one subplot per direction.

import sinter

LINESTYLES = {
    2: ':',
    3: ':',
    4: '--',
    5: '--',
    6: '-',
    7: '-',
}


def _plot_args_func(index, group_key, group_stats):
    m = group_stats[0].json_metadata
    return {'linestyle': LINESTYLES.get(m['expected_distance'], '-')}


# Stable order: named profiles, then mixes, then edges (matches _default_directions).
present = set(df['direction'].unique())
ORDER = ['measurement-limited', 'spam-limited', 'res-limited', 'balanced']
ORDER += sorted(d for d in present if d.startswith('mix_'))
ORDER += sorted(d for d in present if d.startswith('edge_'))
direction_names = [d for d in ORDER if d in present]

n = len(direction_names)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

fits_lookup = fits_df.set_index('direction').to_dict('index') if not fits_df.empty else {}

for idx, direction_name in enumerate(direction_names):
    ax = axes[idx // ncols][idx % ncols]
    grp = df[df['direction'] == direction_name]

    stats = [
        sinter.TaskStats(
            strong_id=row['strong_id'],
            decoder=row['decoder'],
            json_metadata=row['json_metadata'] if isinstance(row['json_metadata'], dict)
                           else ast.literal_eval(row['json_metadata']),
            shots=int(row['shots']),
            errors=int(row['errors']),
            discards=int(row['discards']),
            seconds=float(row['seconds']),
        )
        for _, row in grp.iterrows()
    ]

    sinter.plot_error_rate(
        ax=ax,
        stats=stats,
        x_func=lambda stat: stat.json_metadata['p'],
        group_func=lambda stat: f"d={stat.json_metadata['expected_distance']}",
        plot_args_func=_plot_args_func,
        failure_values_func=lambda stat: stat.json_metadata['k'],
    )

    fit = fits_lookup.get(direction_name)
    if fit is not None:
        ax.axvline(
            fit['p_th'], color='red', linestyle='--', linewidth=1.2,
            label=f"p_th={fit['p_th']:.4f}",
        )

    ax.loglog()
    ax.set_xlabel('Physical Error Rate (p)')
    ax.set_ylabel('Logical Error Rate per round')
    rm = grp['ratio_m_bulk'].iloc[0]
    rs = grp['ratio_spam'].iloc[0]
    rd = grp['ratio_res'].iloc[0]
    ax.set_title(f'{direction_name}\nrm={rm:.2g}, rs={rs:.2g}, rd={rd:.2g}')
    ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.3, alpha=0.5)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc='lower right', frameon=True, edgecolor='black', fancybox=False, fontsize=8)

for idx in range(n, nrows * ncols):
    fig.delaxes(axes[idx // ncols][idx % ncols])

fig.suptitle('Logical error rate per direction (red dashed = fitted p_th)')
fig.tight_layout()
fig.set_dpi(120)
plt.show()

In [ ]:
# 3D surface in (p_m_bulk, p_spam, p_res) space.
#
# Each fitted direction contributes one point P_th = p_th * (r_m, r_s, r_d).
# Triangulate in the 2D simplex (r_m, r_s) -- the third coordinate is
# implicit -- and lift the resulting triangulation to 3D, drawn as a wireframe.

NAMED_PROFILES = {'measurement-limited', 'spam-limited', 'res-limited'} #, 'balanced'}

x = fits_df['p_m_bulk_th'].to_numpy()
y = fits_df['p_spam_th'].to_numpy()
z = fits_df['p_res_th'].to_numpy()
p_th = fits_df['p_th'].to_numpy()

# Normalize ratios to sum=1 for the simplex triangulation (any monotonic
# normalization works; we just need a valid 2D parameterization).
ratios = fits_df[['ratio_m_bulk', 'ratio_spam', 'ratio_res']].to_numpy()
simplex = ratios / ratios.sum(axis=1, keepdims=True)
tri = Triangulation(simplex[:, 0], simplex[:, 1])

fig = plt.figure(figsize=(8, 9))
ax = fig.add_subplot(111, projection='3d')

# Wireframe: transparent faces, visible edges.
wire = ax.plot_trisurf(
    x, y, z,
    triangles=tri.triangles,
    color=(0, 0, 0, 0),
    edgecolor='steelblue',
    linewidth=1.0,
    shade=False,
)

# Sample points at the fitted thresholds.
# ax.scatter(x, y, z, color='steelblue', s=40, edgecolor='k', linewidths=0.4, depthshade=False)

# Highlight the four named profiles for orientation against threshold_comparison.ipynb.
for _, row in fits_df.iterrows():
    if row['direction'] in NAMED_PROFILES:
        ax.scatter(
            row['p_m_bulk_th'], row['p_spam_th'], row['p_res_th'],
            color='red', s=120, marker='*', edgecolor='k', linewidths=0.6, depthshade=False,
        )
        ax.text(
            row['p_m_bulk_th'], row['p_spam_th'], row['p_res_th'],
            f"  {row['direction']}\n  p_th={row['p_th']:.3g}",
            fontsize=8, color='black',
        )

ax.set_xlabel('p_m_bulk (threshold)')
ax.set_ylabel('p_spam (threshold)')
ax.set_zlabel('p_res (threshold)')
# ax.set_title('Planar Honeycomb Threshold surface in flexible_pairwise noise space')

# Axis limits + box aspect (uneven y on purpose).
slack = 1.6
y_slack = 1.1
ax.set_xlim(0, x.max() * slack)
ax.set_ylim(0, y.max() * y_slack)
ax.set_zlim(0, z.max() * slack)
# zoom > 1 makes the 3D content fill more of the axes box, removing the
# blank padding tight_layout leaves around 3D plots.
ax.set_box_aspect(
    (x.max() * slack, y.max() * y_slack, z.max() * slack),
    zoom=1,
)

# Camera in the (+,+,+) octant -> origin sits in the far-back corner.
ax.view_init(elev=20, azim=45)

# tight_layout doesn't handle 3D axes well -- push axes to the figure edges.
# fig.subplots_adjust(left=-10.0, right=1.0, top=0.95, bottom=0.0)
fig.tight_layout()
fig.set_dpi(140)
plt.show()

In [ ]:
# Same wireframe, with values rescaled for a 3-shot majority-vote scheme:
#   measurement: physical p_m -> effective 3 * p_m**2,  so plot sqrt(p_m_bulk_th / 3)
#   residual:    3 measurement rounds -> 3x noise,      so plot p_res_th / 3
#   spam: unchanged
# Labels stay in the original (p_m_bulk, p_spam, p_res) variables.

NAMED_PROFILES = {'measurement-limited', 'spam-limited', 'res-limited'} #, 'balanced'}

x = np.sqrt(fits_df['p_m_bulk_th'].to_numpy() / 3.0)
y = fits_df['p_spam_th'].to_numpy()
z = fits_df['p_res_th'].to_numpy() / 3.0

ratios = fits_df[['ratio_m_bulk', 'ratio_spam', 'ratio_res']].to_numpy()
simplex = ratios / ratios.sum(axis=1, keepdims=True)
tri = Triangulation(simplex[:, 0], simplex[:, 1])

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    x, y, z,
    triangles=tri.triangles,
    color=(0, 0, 0, 0),
    edgecolor='steelblue',
    linewidth=1.0,
    shade=False,
)

for _, row in fits_df.iterrows():
    if row['direction'] in NAMED_PROFILES:
        rx = float(np.sqrt(row['p_m_bulk_th'] / 3.0))
        ry = float(row['p_spam_th'])
        rz = float(row['p_res_th'] / 3.0)
        # Rescaled threshold = distance from origin in the rescaled coordinates.
        # For axis-aligned named profiles this picks out the single nonzero axis:
        #   measurement-limited -> sqrt(p_th/3),  spam-limited -> p_th,
        #   res-limited -> p_th/3.
        p_th_rescaled = float(np.sqrt(rx * rx + ry * ry + rz * rz))
        ax.scatter(rx, ry, rz, color='red', s=120, marker='*', edgecolor='k', linewidths=0.6, depthshade=False)
        ax.text(rx, ry, rz, f"  {row['direction']}\n  p_th={p_th_rescaled:.3g}", fontsize=8, color='black')

ax.set_xlabel('p_m_bulk (threshold)')
ax.set_ylabel('p_spam (threshold)')
ax.set_zlabel('p_res (threshold)')
ax.set_title('Planar Honeycomb Threshold surface (3-shot majority-vote rescaling)')

slack = 1.6
y_slack = 1.1
ax.set_xlim(0, x.max() * slack)
ax.set_ylim(0, y.max() * y_slack)
ax.set_zlim(0, z.max() * slack)
ax.set_box_aspect(
    (x.max() * slack, y.max() * y_slack, z.max() * slack),
    zoom=1.3,
)

ax.view_init(elev=20, azim=45)

fig.subplots_adjust(left=0.0, right=1.0, top=0.95, bottom=0.0)
fig.set_dpi(140)
plt.show()

In [ ]:
# Cross-section at p_spam = 0 (treating ratio_spam ~ 1e-6 as zero).
# Two curves on the same axes:
#   - raw           : (p_m_bulk_th, p_res_th)
#   - majority-vote : (sqrt(p_m_bulk_th / 3), p_res_th / 3)

SPAM_ZERO_TOL = 1e-3

cross = fits_df[fits_df['ratio_spam'] < SPAM_ZERO_TOL].copy()
# Order along the curve by angle from the origin in (p_m_bulk, p_res).
cross['_angle'] = np.arctan2(cross['p_res_th'], cross['p_m_bulk_th'])
cross = cross.sort_values('_angle', ascending=False).reset_index(drop=True)

x_raw = cross['p_m_bulk_th'].to_numpy()
z_raw = cross['p_res_th'].to_numpy()
x_rescaled = np.sqrt(x_raw / 3.0)
z_rescaled = z_raw / 3.0

fig, ax = plt.subplots(figsize=(7, 7))

ax.plot(x_raw, z_raw, '-o', color='steelblue', label='raw')
ax.plot(x_rescaled, z_rescaled, '-o', color='darkorange', label='3-shot majority-vote')

for _, row in cross.iterrows():
    rx_raw = float(row['p_m_bulk_th'])
    rz_raw = float(row['p_res_th'])
    rx_rs = float(np.sqrt(rx_raw / 3.0))
    rz_rs = rz_raw / 3.0
    ax.annotate(row['direction'], (rx_raw, rz_raw), fontsize=8,
                xytext=(5, 5), textcoords='offset points', color='steelblue')
    ax.annotate(row['direction'], (rx_rs, rz_rs), fontsize=8,
                xytext=(5, 5), textcoords='offset points', color='darkorange')

ax.set_xlabel('p_m_bulk (threshold)')
ax.set_ylabel('p_res (threshold)')
ax.set_title('Cross-section at p_spam = 0')

lim = max(x_raw.max(), z_raw.max(), x_rescaled.max(), z_rescaled.max()) * 1.1
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal', adjustable='box')

ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7)
ax.grid(which='minor', linestyle=':', linewidth=0.3, alpha=0.5)
ax.legend(loc='upper right', frameon=True, edgecolor='black', fancybox=False)

fig.tight_layout()
fig.set_dpi(140)
plt.show()

In [11]:
# Optional: persist the fitted thresholds for later use.
out_path = Path('../out/collection/threshold_3d_fits.csv')
fits_df.to_csv(out_path, index=False)
print(f'Saved fits -> {out_path}')

Saved fits -> ../out/collection/threshold_3d_fits.csv
